# RAG Pipeline — Recherche d'Information (RI)
Pipeline : **PDF → extraction → chunking sémantique → embeddings → FAISS → retrieval → génération (Qwen2.5)**

Nouveautés de cette version :
- **Chunking sémantique** : les chunks sont découpés selon le *sens* (changement de sujet), plus à taille fixe.
- **Affichage des sources** : chaque réponse indique de quel document (et quelle page) provient l'information.

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate torch

## 1. Upload des documents

In [ ]:
import os, shutil

DOC_DIR = "data/documents"
os.makedirs(DOC_DIR, exist_ok=True)

try:  # Google Colab : upload manuel
    from google.colab import files
    uploaded = files.upload()      # Ctrl/Shift-clic pour choisir plusieurs fichiers
    for name in uploaded:
        shutil.move(name, os.path.join(DOC_DIR, name))
except ImportError:                # En local : mettre les PDF dans data/documents/
    pass

print("Documents dans le dossier :")
for f in sorted(os.listdir(DOC_DIR)):
    print(" -", f)

Saving 7. Évaluation des performances des systèmes de recherche de RI.pdf to 7. Évaluation des performances des systèmes de recherche de RI (1).pdf
Saving 6. Cour1.pdf to 6. Cour1 (1).pdf
Saving 5. Modele LSI.pdf to 5. Modele LSI (1).pdf
Saving 4. modele p-norme.pdf to 4. modele p-norme (1).pdf
Saving 3.1 difference entre modèle booléen et le modèle vectoriel.pdf to 3.1 difference entre modèle booléen et le modèle vectoriel (1).pdf
Saving 3. modeles-RI-1.pdf to 3. modeles-RI-1 (1).pdf
Saving 2. introduction-RI.pdf to 2. introduction-RI (1).pdf
Saving 1. La définition du domaine RI_chap1.pdf to 1. La définition du domaine RI_chap1 (1).pdf
Documents dans le dossier :
 - 1. La définition du domaine RI_chap1 (1).pdf
 - 2. introduction-RI (1).pdf
 - 3. modeles-RI-1 (1).pdf
 - 3.1 difference entre modèle booléen et le modèle vectoriel (1).pdf
 - 4. modele p-norme (1).pdf
 - 5. Modele LSI (1).pdf
 - 6. Cour1 (1).pdf
 - 7. Évaluation des performances des systèmes de recherche de RI (1).pdf


## 2. Extraction du texte (page par page)

In [ ]:
import re
from pathlib import Path
import pymupdf

def clean_source_name(filename):
    """Retire les suffixes ' (1)' ajoutés par le navigateur lors des uploads multiples."""
    return re.sub(r"\s\(\d+\)(?=\.pdf$)", "", filename)

def extract_pdf_pages(pdf_path):
    pages = []
    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "text": text,
                    "source": clean_source_name(Path(pdf_path).name),
                    "page": page_number,
                })
    return pages

all_pages = []
for pdf_path in sorted(Path(DOC_DIR).glob("*.pdf")):
    all_pages.extend(extract_pdf_pages(pdf_path))

print("Total pages extraites :", len(all_pages))

Total pages extraites : 225


## 3. Modèle d'embeddings
Chargé **avant** le chunking, car le chunking sémantique en a besoin pour mesurer la similarité entre phrases.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (3 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4. Chunking sémantique
Principe :
1. Le texte de chaque page est découpé en **unités** (phrases / lignes de slide).
2. Chaque unité est transformée en embedding.
3. On ajoute les unités une à une au chunk courant. On **ouvre un nouveau chunk** quand :
   - la similarité cosinus entre l'unité et le centre (moyenne) du chunk courant est **< `similarity_threshold`** (changement de sujet), ou
   - le chunk dépasserait **`max_chars`**.
4. Un chunk n'est jamais coupé avant d'avoir **`min_chars`** (évite les mini-chunks).

> `similarity_threshold` : plus il est **haut**, plus les chunks sont **petits** (plus de coupures). À ajuster selon vos cours (0.35 – 0.6).

In [ ]:
import numpy as np

def split_into_units(text, min_unit_chars=40):
    """Découpe en phrases/lignes, puis fusionne les unités trop courtes avec la suivante."""
    text = re.sub(r"[ \t]+", " ", text)
    raw = [u.strip() for u in re.split(r"(?<=[.!?])\s+|\n+", text) if u.strip()]

    units, buffer = [], ""
    for u in raw:
        buffer = f"{buffer} {u}".strip()
        if len(buffer) >= min_unit_chars:
            units.append(buffer)
            buffer = ""
    if buffer:
        if units:
            units[-1] = f"{units[-1]} {buffer}"
        else:
            units.append(buffer)
    return units


def semantic_chunk(text, model, similarity_threshold=0.45,
                   min_chars=250, max_chars=1200):
    units = split_into_units(text)
    if not units:
        return []
    if len(units) == 1:
        return units

    vectors = model.encode(units, convert_to_numpy=True,
                           normalize_embeddings=True, show_progress_bar=False)

    chunks = []
    current_units = [units[0]]
    current_vecs = [vectors[0]]
    current_len = len(units[0])

    for unit, vec in zip(units[1:], vectors[1:]):
        centroid = np.mean(current_vecs, axis=0)
        centroid /= (np.linalg.norm(centroid) + 1e-9)
        sim = float(np.dot(centroid, vec))

        topic_change = sim < similarity_threshold and current_len >= min_chars
        too_long = current_len + len(unit) + 1 > max_chars

        if topic_change or too_long:
            chunks.append(" ".join(current_units))
            current_units, current_vecs, current_len = [unit], [vec], len(unit)
        else:
            current_units.append(unit)
            current_vecs.append(vec)
            current_len += len(unit) + 1

    chunks.append(" ".join(current_units))
    return chunks

In [ ]:
all_chunks = []
for page in all_pages:
    page_chunks = semantic_chunk(page["text"], embedding_model,
                                 similarity_threshold=0.45,
                                 min_chars=250, max_chars=1200)
    for chunk_id, chunk_text in enumerate(page_chunks):
        all_chunks.append({
            "text": chunk_text,
            "source": page["source"],
            "page": page["page"],
            "chunk_id": chunk_id,
        })

lengths = [len(c["text"]) for c in all_chunks]
print("Total chunks :", len(all_chunks))
print(f"Taille des chunks (caractères) : min={min(lengths)}  moyenne={int(np.mean(lengths))}  max={max(lengths)}")

Total chunks : 393
Taille des chunks (caractères) : min=43  moyenne=321  max=1198


## 5. Embeddings des chunks + index FAISS

In [ ]:
import faiss, json

texts = [c["text"] for c in all_chunks]
embeddings = embedding_model.encode(
    texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
).astype("float32")
print("Embedding shape :", embeddings.shape)

index = faiss.IndexFlatIP(embeddings.shape[1])   # produit scalaire = cosinus (vecteurs normalisés)
index.add(embeddings)
print("Vecteurs indexés :", index.ntotal)

faiss.write_index(index, "rag_index.faiss")
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embedding shape : (393, 384)
Vecteurs indexés : 393


## 6. Retrieval

In [ ]:
def retrieve(question, top_k=4):
    query_vector = embedding_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, positions = index.search(query_vector, top_k)

    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos == -1:
            continue
        item = dict(all_chunks[pos])
        item["score"] = float(score)
        results.append(item)
    return results

## 7. Modèle de génération

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 8. Question → Réponse **avec sources**
Chaque passage envoyé au LLM est étiqueté `[Source n]`. Après la réponse, on affiche la liste des documents/pages utilisés avec le score de similarité.

In [ ]:
def build_context(docs):
    blocks = []
    for i, d in enumerate(docs, start=1):
        blocks.append(f"[Source {i} — {d['source']}, page {d['page']}]\n{d['text']}")
    return "\n\n".join(blocks)


def display_sources(docs, preview_chars=220):
    print("\n" + "=" * 70)
    print("📚 SOURCES UTILISÉES")
    print("=" * 70)
    for i, d in enumerate(docs, start=1):
        preview = d["text"][:preview_chars].replace("\n", " ")
        print(f"[Source {i}] 📄 {d['source']}  |  page {d['page']}  |  score = {d['score']:.3f}")
        print(f"           « {preview}… »\n")


def rag_answer(query, top_k=3, max_new_tokens=300, show_sources=True):
    docs = retrieve(query, top_k=top_k)
    context = build_context(docs)

    messages = [
        {"role": "system", "content": (
            "Tu es un assistant pédagogique spécialisé en Recherche d'Information (RI). "
            "Réponds à la question en t'appuyant uniquement sur le contexte fourni. "
            "Cite les sources utilisées sous la forme [Source n]. "
            "Si la réponse n'est pas dans le contexte, dis-le clairement.\n\n"
            f"Contexte :\n{context}")},
        {"role": "user", "content": query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print("❓ Question :", query)
    print("\n💬 Réponse :\n" + answer)
    if show_sources:
        display_sources(docs)
    return answer, docs

In [ ]:
answer, docs = rag_answer("Qu'est-ce que le modèle vectoriel ?")

❓ Question : Qu'est-ce que le modèle vectoriel ?

💬 Réponse :
Le modèle vectoriel est l'un des deux modèles classiques de Recherche d'Information (RI) les plus connus, l'autre étant le modèle booléen. Il repose sur des philosophies mathématiques et des logiques d'appariement distincts des modèles booléens. 

Selon [Source 2], ce modèle utilise des représentations vectorielles des documents et des termes, permettant ainsi une analyse plus fine et complexe des données. Il s'appuie sur des techniques de calcul vectoriel pour mesurer la similarité entre documents et termes, offrant une approche plus quantitative et sophistiquée par rapport au modèle booléen qui se base sur des opérations de recherche de mots clés.

📚 SOURCES UTILISÉES
[Source 1] 📄 5. Modele LSI.pdf  |  page 7  |  score = 0.645
           « Ce vecteur est déjà de norme 1. 6.3. Troisième vecteur propre Pour , nous résolvons : D'où : Nous choisissons , puis normalisons le vecteur obtenu. Cela donne : Les vecteurs sont orthono

In [ ]:
ma_question = input()
answer, docs = rag_answer(ma_question, top_k=3)

❓ Question : Comment évalue-t-on les performances d'un système de recherche d'information 

💬 Réponse :
Les performances d'un système de recherche d'information peuvent être évaluées à travers plusieurs indicateurs et dimensions, comme indiqué dans le contexte fourni. Voici les principaux aspects d'évaluation :

1. **Indicateurs de Pertinence** : Ces indicateurs mesurent la capacité du système à trouver des documents pertinents. Ils peuvent être établis par des évaluateurs humains ou fournies par des jeux de données de référence.

2. **Efficacité Documentaire** : Cette dimension évalue la capacité du système à retrouver les documents pertinents. Elle mesure comment bien le système trouve les documents qui répondent aux besoins de l'utilisateur.

3. **Qualité du Classement** : Cette dimension évalue la capacité du système à classer les documents pertinents en ordre de priorité. Elle mesure comment bien le système place les documents les plus pertinents en haut de la liste de résultats.
